# Build Android APK with Buildozer on Google Colab

This notebook compiles your Kivy app into an Android APK.

**Instructions:**
1. Click 'Runtime' → 'Run all'
2. Wait 30-60 minutes
3. Download the APK from the final step

**Note:** If re-running, first do: Runtime → Restart runtime...

In [ ]:
# Step 1: Install system dependencies
import subprocess
import sys
import os

print("Installing system dependencies...")
subprocess.run(["apt", "update"], check=True)
subprocess.run([
    "apt", "install", "-y",
    "python3-pip", "python3-dev", "python3-venv", "git", "zip", "unzip",
    "openjdk-17-jdk", "libbz2-dev", "libncurses5-dev", "libffi-dev",
    "libreadline-dev", "libsqlite3-dev", "zlib1g-dev", "liblzma-dev",
    "autoconf", "libtool", "pkg-config", "python3-setuptools",
    "wget", "curl", "build-essential"
], check=True)

print("\nInstalling Cython and Buildozer...")
subprocess.run([sys.executable, "-m", "pip", "install", "--user", "cython==0.29.19"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--user", "buildozer"], check=True)

print("\nSetup complete!")

In [ ]:
# Step 2: Patch buildozer source file on disk (skip root check)
# This is the ONLY reliable way - patch the actual file the subprocess will load.
import os
import re

# Find buildozer package location
import buildozer
pkg_dir = os.path.dirname(buildozer.__file__)
init_path = os.path.join(pkg_dir, "__init__.py")
print(f"Patching buildozer source: {init_path}")

with open(init_path, 'r') as f:
    content = f.read()

# Replace the entire check_root method body with a no-op
# Match from 'def check_root(self):' up to the next method definition
pattern = r'(\n    def check_root\(self\):\n)((?:.*\n)*?)(?=\n    def |\nclass )'

replacement = '''\n    def check_root(self):\n        """Check if the user is root, and warn if so."""\n        return  # PATCHED: skip root check\n\n'''

new_content = re.sub(pattern, replacement, content, count=1, flags=re.DOTALL)

if new_content == content:
    print("WARNING: Could not patch via regex, trying string replace...")
    # Fallback: simpler approach
    lines = content.split('\n')
    new_lines = []
    in_check_root = False
    for line in lines:
        if 'def check_root(self):' in line:
            in_check_root = True
            new_lines.append(line)
            new_lines.append('        """Check if the user is root, and warn if so."""')
            new_lines.append('        return  # PATCHED: skip root check')
            continue
        if in_check_root:
            # Skip all lines until we reach the next method (4-space indent + 'def ')
            if line.startswith('    def ') or line.startswith('class '):
                in_check_root = False
            else:
                continue
        new_lines.append(line)
    new_content = '\n'.join(new_lines)

with open(init_path, 'w') as f:
    f.write(new_content)

print("✅ Patched buildozer source file successfully!")

# Verify the patch
with open(init_path, 'r') as f:
    patched = f.read()
if 'return  # PATCHED: skip root check' in patched:
    print("✅ Patch verified!")
else:
    print("❌ Patch may have failed, proceeding anyway...")

In [ ]:
# Step 3: Clone repository and enter android directory
import os
import subprocess
import shutil

repo_dir = "network-scanner"
if os.path.exists(repo_dir):
    print(f"Directory '{repo_dir}' already exists. Removing and re-cloning...")
    shutil.rmtree(repo_dir)

print("Cloning repository...")
subprocess.run(["git", "clone", "https://github.com/Loongood666/network-scanner.git"], check=True)
os.chdir("network-scanner/android")

print("\nCurrent directory:")
subprocess.run(["pwd"], check=True)
print("\nFiles in android directory:")
subprocess.run(["ls", "-la"], check=True)

In [ ]:
# Step 4: Build APK (this will take 30-60 minutes)
import subprocess
import sys
import os

print("Starting APK build...")
print("This will take 30-60 minutes. Please be patient.")
print("You can leave this tab open and come back later.\n")

# Ensure ~/.local/bin is in PATH
local_bin = os.path.expanduser("~/.local/bin")
if local_bin not in os.environ["PATH"]:
    os.environ["PATH"] = local_bin + ":" + os.environ["PATH"]

# Run buildozer (check_root is already patched in the source file)
result = subprocess.run(
    ["buildozer", "-v", "android", "debug"],
    capture_output=False,
    text=True,
    timeout=None
)

if result.returncode == 0:
    print("\n✅ Build completed successfully!")
else:
    print(f"\n❌ Build failed with return code {result.returncode}")
    sys.exit(1)

In [ ]:
# Step 5: Download the APK
import os
from google.colab import files

apk_dir = "bin"
if os.path.exists(apk_dir):
    apks = [f for f in os.listdir(apk_dir) if f.endswith(".apk")]
    if apks:
        print(f"✅ Found APK: {apks[0]}")
        files.download(os.path.join(apk_dir, apks[0]))
    else:
        print("❌ No APK found. Build may have failed.")
        print("\nChecking bin directory:")
        for f in os.listdir(apk_dir):
            print(f)
else:
    print("❌ Build directory 'bin' not found.")
    print("\nChecking current directory...")
    import subprocess
    subprocess.run(["ls", "-la"], check=False)